### Libraries 

In [1]:
import sys
sys.path.append(r'C:\Users\User\005_Libraries')

import mylibs

In [2]:
!pip install anthropic


[notice] A new release of pip is available: 24.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import anthropic
import json
import os
from dotenv import load_dotenv
from glob import glob

In [11]:
import time
from datetime import datetime

### Load the most recent CSV

In [4]:
csv_files= glob('002_Data/gaming_tweets_*.csv')

if csv_files:
    latest_file= sorted(csv_files)[-1]
    
    df_tweets_100= pd.read_csv(latest_file)
    
    print(f"Loaded {len(df_tweets_100)} tweets from: {latest_file}")
    print(f"\nData preview:")
    print(df_tweets_100.head())
    print(f"\nColumns: {list(df_tweets_100.columns)}")
else:
    print("No CSV files found!")

Loaded 100 tweets from: 002_Data\gaming_tweets_20251218_142234-Copy1.csv

Data preview:
              tweet_id                                               text  \
0  2001656770645615013             @Lexar_Gaming I need one @jonanarranzz   
1  2001656753029550085  Cleared the bs reposts off my profile. Kept th...   
2  2001656732943040516  @OBE1plays That would be all of my favourite g...   
3  2001656725246521571  The most dangerous player in efootball \n#eFoo...   
4  2001656700269445349           @CommodoreBlog This was peak C64 gaming.   

                  created_at            author_id lang  retweet_count  \
0  2025-12-18 14:12:17+00:00  1514991564174635012   en              0   
1  2025-12-18 14:12:13+00:00  1880073685794447360   en              0   
2  2025-12-18 14:12:08+00:00  1122515293870231553   en              0   
3  2025-12-18 14:12:06+00:00  1491794006031802368   en              0   
4  2025-12-18 14:12:00+00:00             19237341   en              0   

   reply_c

### Set Up Claude API

In [5]:
load_dotenv()
claude_api_key= os.getenv('ANTHROPIC_API_KEY')

if claude_api_key:
    print("Claude API key found!")
        
    try:
        client= anthropic.Anthropic(api_key= claude_api_key)
        message= client.messages.create(model= "claude-sonnet-4-20250514",
                                        max_tokens= 100,
                                        messages= [{"role": "user", 
                                                    "content": "Say 'Hello, I'm ready to analyze gaming tweets!'"
                                                     }]
                                        )
        
        response= message.content[0].text
        print(f"\nSUCCESS! Claude says: {response}")
                
    except Exception as e:
        print(f"\nConnection error: {e}")
else:
    print("Claude API key still not found!")

Claude API key found!

SUCCESS! Claude says: Hello, I'm ready to analyze gaming tweets!


### Analysis Function

In [6]:
def analyze_tweet(tweet_text):
    """
    Analyze a gaming tweet with Claude AI
    
    Args:
        tweet_text (str): The tweet text to analyze
        
    Returns:
        dict: Analysis with sentiment, topics, entities
    """
    
    prompt= f"""Analyze this gaming tweet and provide structured output in JSON format

Tweet: "{tweet_text}"

Provide analysis in this EXACT JSON format (no additional text):
{{
    "sentiment": "positive/negative/neutral",
    "sentiment_score": 0.0 to 1.0 (0=very negative, 0.5=neutral, 1=very positive),
    "primary_topic": "main topic in 2-3 words",
    "topics": ["topic1", "topic2", "topic3"],
    "entities": {{
        "games": ["game1", "game2"],
        "companies": ["company1"],
        "platforms": ["platform1"],
        "people": ["person1"]
                }},
    "content_type": "complaint/excitement/question/news/review/discussion/other",
    "summary": "one sentence summary"
}}

Return ONLY valid JSON, no markdown, no extra text"""

    try:
        message= client.messages.create(
            model= "claude-sonnet-4-20250514",
            max_tokens= 1000,
            messages= [{"role": "user", 
                        "content": prompt}
                      ])
        
        response_text= message.content[0].text
        
        response_text= response_text.replace('```json', '').replace('```', '').strip()
        
        analysis= json.loads(response_text)
        
        return analysis
        
    except json.JSONDecodeError as e:
        print(f"JSON parsing error for tweet: {tweet_text[:50]}...")
        return {
            "sentiment": "neutral",
            "sentiment_score": 0.5,
            "primary_topic": "unknown",
            "topics": [],
            "entities": {"games": [], "companies": [], "platforms": [], "people": []},
            "content_type": "other",
            "summary": "Analysis unavailable"
               }
        
    except Exception as e:
        print(f"API error: {e}")
        return {
            "sentiment": "neutral",
            "sentiment_score": 0.5,
            "primary_topic": "unknown",
            "topics": [],
            "entities": {"games": [], "companies": [], "platforms": [], "people": []},
            "content_type": "other",
            "summary": "Analysis unavailable"
               }

print("✅ Analysis function created!")

✅ Analysis function created!


### Test with ONE Tweet

In [7]:
test_tweet= df_tweets_100.iloc[0]['text']

print(f"Tweet: {test_tweet[:100]}...")
result= analyze_tweet(test_tweet)

print(json.dumps(result, indent= 2))

Tweet: @Lexar_Gaming I need one @jonanarranzz...
{
  "sentiment": "positive",
  "sentiment_score": 0.7,
  "primary_topic": "product request",
  "topics": [
    "product request",
    "gaming hardware",
    "social interaction"
  ],
  "entities": {
    "games": [],
    "companies": [
      "Lexar Gaming"
    ],
    "platforms": [],
    "people": [
      "jonanarranzz"
    ]
  },
  "content_type": "other",
  "summary": "User expressing need for a Lexar Gaming product while tagging another person."
}


#### Analyze ALL 100 Tweets

In [12]:
all_results= []

start_time= time.time()

for i, row in df_tweets_100.iterrows():
    tweet_text= row['text']
    
    if (i+1)%10== 0 or i== 0:
        print(f"Processing tweet {i+1}/100...")
    
    analysis= analyze_tweet(tweet_text)
    result= {
        'tweet_id': row['tweet_id'],
        'text': row['text'],
        'created_at': row['created_at'],
        'author_id': row['author_id'],
        'retweet_count': row['retweet_count'],
        'reply_count': row['reply_count'],
        'like_count': row['like_count'],
        'quote_count': row['quote_count'],
        'sentiment': analysis['sentiment'],
        'sentiment_score': analysis['sentiment_score'],
        'primary_topic': analysis['primary_topic'],
        'topics': str(analysis['topics']),
        'games': str(analysis['entities']['games']),
        'companies': str(analysis['entities']['companies']),
        'platforms': str(analysis['entities']['platforms']),
        'people': str(analysis['entities']['people']),
        'content_type': analysis['content_type'],
        'summary': analysis['summary'],
        'analyzed_at': datetime.now()
            }
    
    all_results.append(result)
    
    time.sleep(0.5)

end_time= time.time()
duration= (end_time - start_time)/60

print(f"Analyzed {len(all_results)} tweets!")
print(f"Time taken: {duration:.2f} minutes")
print(f"Estimated cost: ~${len(all_results) * 0.01:.2f}")

Processing tweet 1/100...
Processing tweet 10/100...
Processing tweet 20/100...
Processing tweet 30/100...
Processing tweet 40/100...
Processing tweet 50/100...
Processing tweet 60/100...
Processing tweet 70/100...
Processing tweet 80/100...
Processing tweet 90/100...
Processing tweet 100/100...
Analyzed 100 tweets!
Time taken: 7.02 minutes
Estimated cost: ~$1.00


In [13]:
df_analyzed= pd.DataFrame(all_results)

print(f"Rows: {len(df_analyzed)}")
print(f"Columns: {len(df_analyzed.columns)}")
print(df_analyzed.head())

Rows: 100
Columns: 19
              tweet_id                                               text  \
0  2001656770645615013             @Lexar_Gaming I need one @jonanarranzz   
1  2001656753029550085  Cleared the bs reposts off my profile. Kept th...   
2  2001656732943040516  @OBE1plays That would be all of my favourite g...   
3  2001656725246521571  The most dangerous player in efootball \n#eFoo...   
4  2001656700269445349           @CommodoreBlog This was peak C64 gaming.   

                  created_at            author_id  retweet_count  reply_count  \
0  2025-12-18 14:12:17+00:00  1514991564174635012              0            0   
1  2025-12-18 14:12:13+00:00  1880073685794447360              0            0   
2  2025-12-18 14:12:08+00:00  1122515293870231553              0            0   
3  2025-12-18 14:12:06+00:00  1491794006031802368              0            0   
4  2025-12-18 14:12:00+00:00             19237341              0            0   

   like_count  quote_count s

In [17]:
df_analyzed

,tweet_id,text,created_at,author_id,retweet_count,reply_count,like_count,quote_count,sentiment,sentiment_score,primary_topic,topics,games,companies,platforms,people,content_type,summary,analyzed_at
0,2001656770645615013,@Lexar_Gaming I need one @jonanarranzz,2025-12-18 14:12:17+00:00,1514991564174635012,0,0,0,0,positive,0.70,product request,"['gaming hardware', 'product desire', 'social ...",[],['Lexar Gaming'],[],['jonanarranzz'],other,User expresses need for a Lexar Gaming product...,2026-01-10 11:35:17.635985
1,2001656753029550085,Cleared the bs reposts off my profile. Kept th...,2025-12-18 14:12:13+00:00,1880073685794447360,0,0,0,0,neutral,0.60,profile cleanup,"['social media management', 'content curation'...",[],[],[],[],other,User cleaned their social media profile by rem...,2026-01-10 11:35:21.558546
2,2001656732943040516,@OBE1plays That would be all of my favourite g...,2025-12-18 14:12:08+00:00,1122515293870231553,0,0,0,0,positive,0.90,gaming preferences,"['gaming companies', 'favorite games', 'brand ...",[],"['Nintendo', 'Square Enix']",[],['OBE1plays'],discussion,User expresses strong preference for Nintendo ...,2026-01-10 11:35:25.692536
3,2001656725246521571,The most dangerous player in efootball \n#eFoo...,2025-12-18 14:12:06+00:00,1491794006031802368,0,0,0,0,positive,0.70,eFootball gameplay,"['eFootball skills', 'player performance', 'fo...","['eFootball', 'EA FC', 'UFL']",[],['PS5'],[],excitement,Player boasting about being the most dangerous...,2026-01-10 11:35:30.150225
4,2001656700269445349,@CommodoreBlog This was peak C64 gaming.,2025-12-18 14:12:00+00:00,19237341,0,0,0,0,positive,0.80,retro gaming,"['retro gaming', 'C64', 'nostalgia']",[],['Commodore'],['C64'],[],discussion,User expresses nostalgia about Commodore 64 ga...,2026-01-10 11:35:33.981522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2001655584056316400,@0xCrocy @Kindred_AI kindred ai turning solo g...,2025-12-18 14:07:34+00:00,1097806680375492609,0,0,1,0,positive,0.80,AI gaming companion,"['AI gaming', 'multiplayer experience', 'gamin...",[],['Kindred_AI'],[],['0xCrocy'],excitement,User expresses enthusiasm about Kindred AI's a...,2026-01-10 11:41:58.460309
96,2001655577785893171,DM now for 2D/3D Vtuber Model!\n\nNow availabl...,2025-12-18 14:07:33+00:00,1887980906914398209,0,0,0,0,positive,0.70,VTuber Services,"['vtuber model creation', 'streaming services'...",[],[],['Twitch'],[],other,User is advertising their services for creatin...,2026-01-10 11:42:02.927867
97,2001655571490213979,@Arlo140f3qz Useful gaming guide.,2025-12-18 14:07:31+00:00,1786799937935589377,0,0,0,0,positive,0.70,gaming guide,"['gaming guide', 'recommendation', 'utility']",[],[],[],['Arlo140f3qz'],review,User positively acknowledges a gaming guide as...,2026-01-10 11:42:06.557912
98,2001655556130705499,"In-game assets are meant to be used, not flipp...",2025-12-18 14:07:28+00:00,342153773,0,0,0,0,positive,0.75,Web3 gaming,"['Web3 gaming', 'NFT marketplace', 'game econo...",[],"['Immutable', 'GBMauction']",['Immutable'],[],discussion,User advocates for GBM Auction House on Immuta...,2026-01-10 11:42:10.907881


### Save Results to CSV

In [20]:
timestamp= datetime.now().strftime('%Y%m%d_%H%M%S')
filename= f'002_Data/gaming_tweets_analyzed_{timestamp}.csv'

df_analyzed.to_csv(filename, index= False, encoding= 'utf-8-sig')

### Alternative way using Src

In [ ]:
from 001_Src.claude_analyzer import analyze_tweet

all_results= []

start_time= time.time()

for i, row in df_tweets_100.iterrows():
    tweet_text= row['text']
    
    if (i+1)%10== 0 or i== 0:
        print(f"Processing tweet {i+1}/100...")
    
    analysis= analyze_tweet(tweet_text)
    result= {
        'tweet_id': row['tweet_id'],
        'text': row['text'],
        'created_at': row['created_at'],
        'author_id': row['author_id'],
        'retweet_count': row['retweet_count'],
        'reply_count': row['reply_count'],
        'like_count': row['like_count'],
        'quote_count': row['quote_count'],
        'sentiment': analysis['sentiment'],
        'sentiment_score': analysis['sentiment_score'],
        'primary_topic': analysis['primary_topic'],
        'topics': str(analysis['topics']),
        'games': str(analysis['entities']['games']),
        'companies': str(analysis['entities']['companies']),
        'platforms': str(analysis['entities']['platforms']),
        'people': str(analysis['entities']['people']),
        'content_type': analysis['content_type'],
        'summary': analysis['summary'],
        'analyzed_at': datetime.now()
            }
    
    all_results.append(result)
    
    time.sleep(0.5)

end_time= time.time()
duration= (end_time - start_time)/60

print(f"Analyzed {len(all_results)} tweets!")
print(f"Time taken: {duration:.2f} minutes")
print(f"Estimated cost: ~${len(all_results) * 0.01:.2f}")